# Model D — Risk & Priority Scoring

Scores patient urgency for clinician review. No diagnosis. Output is clinician-facing only.
No training data required — uses Gemini with explicit triage rules.

**Output schema:**
```json
{
  "priority": "HIGH | MEDIUM | LOW",
  "suspected_issue": "<broad category>",
  "risk_factors": ["..."],
  "confidence": 0.0
}
```

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import json
import re
from dataclasses import dataclass, field, asdict
from typing import List, Optional

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=genai.types.GenerationConfig(temperature=0.0, max_output_tokens=256),
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"},
    ],
)

In [ ]:
@dataclass
class SymptomInput:
    symptom:             str            = ""
    severity:            str            = ""
    duration:            str            = ""
    body_part:           str            = ""
    associated_symptoms: List[str]      = field(default_factory=list)
    red_flags_present:   Optional[bool] = None
    age:                 Optional[int]  = None


@dataclass
class RiskScore:
    priority:        str       = "MEDIUM"
    suspected_issue: str       = ""
    risk_factors:    List[str] = field(default_factory=list)
    confidence:      float     = 0.0


PRIORITY_LEVELS = ["HIGH", "MEDIUM", "LOW"]

ISSUE_CATEGORIES = [
    "cardiorespiratory-related complaint",
    "neurological-related complaint",
    "gastrointestinal-related complaint",
    "musculoskeletal-related complaint",
    "dermatological-related complaint",
    "urological-related complaint",
    "ENT-related complaint",
    "general or systemic complaint",
    "mental health-related complaint",
    "obstetric or gynaecological complaint",
    "paediatric-related complaint",
    "unclear or unclassifiable complaint",
]

In [ ]:
TRIAGE_RULES = """\
HIGH: red flag present, OR severity severe/extreme/8-10, OR acute rapid onset, OR infant/elderly with moderate+ symptoms.
MEDIUM: moderate severity (4-7/10), symptom hours to days, mild associated symptoms, no red flags.
LOW: mild severity (1-3/10), stable for days to weeks, no associated symptoms, no red flags.

Confidence: 0.8-1.0 if severity + duration + red flag status all known. 0.5-0.79 if one is missing. Below 0.5 if most are missing.
Risk factors: list only observable facts from the data. Max 4. Short phrases only.
"""

SYSTEM_PROMPT = """\
You are a clinical triage scoring assistant. Output a risk score for clinician review only.
Rules:
- No diagnosis, no treatment advice, no patient-facing language.
- Output valid JSON only matching the schema. No extra text.
- suspected_issue must be one value from the provided category list.
"""


def build_prompt(inp: SymptomInput) -> str:
    data = {
        "symptom":             inp.symptom or "unknown",
        "severity":            inp.severity or "unknown",
        "duration":            inp.duration or "unknown",
        "body_part":           inp.body_part or "unknown",
        "associated_symptoms": inp.associated_symptoms,
        "red_flags_present":   inp.red_flags_present,
        "age":                 inp.age,
    }
    schema = {"priority": "HIGH|MEDIUM|LOW", "suspected_issue": "<one from list>",
              "risk_factors": ["<fact>"], "confidence": 0.0}
    categories = "\n".join(f"- {c}" for c in ISSUE_CATEGORIES)

    return f"""\
Patient data:
{json.dumps(data, indent=2)}

Allowed categories:
{categories}

Triage rules:
{TRIAGE_RULES}

Schema:
{json.dumps(schema, indent=2)}

Return the populated JSON only."""

In [ ]:
def _parse_score(raw: str) -> RiskScore:
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
    match   = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("No JSON in response.")
    d = json.loads(match.group(0))

    priority = str(d.get("priority", "MEDIUM")).upper()
    if priority not in PRIORITY_LEVELS:
        priority = "MEDIUM"

    suspected = d.get("suspected_issue", "")
    if suspected not in ISSUE_CATEGORIES:
        suspected = "unclear or unclassifiable complaint"

    factors = d.get("risk_factors", [])
    if not isinstance(factors, list):
        factors = [str(factors)] if factors else []

    try:
        conf = round(max(0.0, min(1.0, float(d.get("confidence", 0.5)))), 2)
    except (TypeError, ValueError):
        conf = 0.5

    return RiskScore(priority=priority, suspected_issue=suspected,
                     risk_factors=factors[:4], confidence=conf)


def score(symptom_input: SymptomInput) -> dict:
    """
    Score patient urgency from structured symptom data.
    Returns dict with 'score' (RiskScore) and 'score_dict'.
    """
    # Safety override: confirmed red flag + severe -> always HIGH
    if symptom_input.red_flags_present and symptom_input.severity.lower() in (
        "severe", "extreme", "unbearable", "very severe"
    ):
        result = RiskScore(
            priority        = "HIGH",
            suspected_issue = "cardiorespiratory-related complaint"
                              if any(w in symptom_input.symptom.lower() for w in ["chest", "breath", "heart"])
                              else "general or systemic complaint",
            risk_factors    = ([symptom_input.symptom] + symptom_input.associated_symptoms)[:4],
            confidence      = 0.95,
        )
        return {"score": result, "score_dict": asdict(result), "success": True, "error": None}

    chat = model.start_chat(history=[
        {"role": "user",  "parts": [SYSTEM_PROMPT]},
        {"role": "model", "parts": ["Understood. JSON risk score only."]},
    ])

    try:
        response = chat.send_message(build_prompt(symptom_input))
        result   = _parse_score(response.text)
        return {"score": result, "score_dict": asdict(result), "success": True, "error": None}
    except Exception as e:
        fallback = RiskScore(priority="MEDIUM", suspected_issue="unclear or unclassifiable complaint")
        return {"score": fallback, "score_dict": asdict(fallback), "success": False, "error": str(e)}


def score_from_model_b(model_b_output: dict, age: Optional[int] = None) -> dict:
    """Convenience wrapper that accepts Model B's output dict directly."""
    ext = model_b_output.get("extraction_dict", {})
    return score(SymptomInput(
        symptom             = ext.get("chief_complaint", ""),
        severity            = ext.get("severity", ""),
        duration            = ext.get("duration", ""),
        body_part           = ext.get("body_part", ""),
        associated_symptoms = ext.get("associated_symptoms", []),
        red_flags_present   = ext.get("red_flags_present"),
        age                 = age,
    ))

---
## Usage

### Option A — Direct input

In [ ]:
result = score(SymptomInput(
    symptom="chest pain", severity="severe", duration="2 hours",
    body_part="chest", associated_symptoms=["shortness of breath"],
    red_flags_present=True, age=54,
))
print(json.dumps(result["score_dict"], indent=2))

### Option B — From Model B output

In [ ]:
# result = score_from_model_b(model_b_result, age=54)
# print(json.dumps(result["score_dict"], indent=2))